# P4-S4 — HerBERT fine-tuning (encoder + two classification heads)

Fine-tunes `allegro/herbert-base-cased` (Polish BERT encoder, CC BY 4.0) into two separate classifiers — `sentiment` (3 classes) and `event_type` (13 classes) — on the P4 news-classifier corpus. `tickers` is **not** learned by the network: extracted deterministically from `TICKER_CATALOG`, same approach as the TF-IDF+LogReg baseline (REQ-002 forbids guessing a ticker; a classifier could hallucinate one).

**Runtime:** `Runtime` → `Change runtime type` → `T4 GPU`, before running any cell below.

**Caveat (2026-09-19):** the corpus has 92 labeled headlines (86 after dedup), far below the 300-500 target — REQ-011's kappa calibration also came back below the 0.6 bar (owner accepted, see `docs/ROADMAP.md`). This run is a **pipeline sanity check**, not a real quality signal. Re-run once the corpus has grown (it grows daily via the Windows Task Scheduler job, see `scripts/news_classifier_run_labeling.ps1`) before trusting any number this notebook prints.

**Zero cost:** no LLM API calls here — only local (Colab GPU) training. CLAUDE.md rule 9 doesn't apply to this notebook.

## 1. Clone the repo and install

In [ ]:
import os
import sys

# Absolute path + directory check makes this cell safe to run more than
# once in the same session (e.g. after Runtime > Run all) -- a relative
# `%cd fin-ai-lab` run twice nests a second clone inside the first
# (found live: /content/fin-ai-lab/fin-ai-lab, pip installing the wrong
# copy and leaving `import fin_ai_lab` unresolved).
REPO_DIR = "/content/fin-ai-lab"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/devTomaszStoklosa/fin-ai-lab.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -e . -q
!pip install transformers datasets scikit-learn -q

# `pip install -e .` alone isn't enough in a session that's already
# running (found live): editable installs add a .pth file that Python's
# import system only reads at interpreter startup, not mid-session, so
# `import fin_ai_lab` fails right after installing without this. Explicit
# sys.path entry sidesteps that caching entirely -- reliable regardless
# of pip's editable-install internals.
SRC_DIR = f"{REPO_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

## 2. Get the corpus onto this Colab session

The corpus lives at `data/corpus/news_classifier/labeled.jsonl` on your own machine (gitignored — Colab has no access to it). Before running the next cell:

1. Upload that file to your Google Drive, e.g. `My Drive/fin-ai-lab/labeled.jsonl`.
2. Update `CORPUS_PATH` below to match where you put it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CORPUS_PATH = "/content/drive/MyDrive/fin-ai-lab/labeled.jsonl"
CHECKPOINT_DIR = "/content/drive/MyDrive/fin-ai-lab/checkpoints/herbert"

## 3. Load and split the corpus

Reuses the repo's own `corpus_store`/`split` code — same chronological split, same near-duplicate dedup (REQ-005) as everywhere else in P4, not reimplemented here.

In [ ]:
from pathlib import Path

from fin_ai_lab.news_classifier.corpus_store import load_labeled
from fin_ai_lab.news_classifier.split import split_chronological

corpus = load_labeled(Path(CORPUS_PATH))
print(f"Loaded {len(corpus)} labeled headlines")

train_items, dev_items, test_items = split_chronological(corpus)
print(f"train={len(train_items)} dev={len(dev_items)} test={len(test_items)}")

## 4. Build HF datasets and tokenize

Label lists come from `models.py`'s own `Literal` types via `typing.get_args` — single source of truth, so this notebook never hardcodes a category list that could drift from the real schema.

In [ ]:
from typing import get_args

from datasets import Dataset
from transformers import AutoTokenizer

from fin_ai_lab.news_classifier.models import EventType, LabeledHeadline, Sentiment

SENTIMENTS = list(get_args(Sentiment))
EVENT_TYPES = list(get_args(EventType))

MODEL_ID = "allegro/herbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def _text(item: LabeledHeadline) -> str:
    return f"{item.headline.headline} {item.headline.lead or ''}".strip()


def _to_dataset(items: list[LabeledHeadline], label_field: str, label_values: list[str]) -> Dataset:
    label_to_id = {value: i for i, value in enumerate(label_values)}
    data = {
        "text": [_text(item) for item in items],
        "label": [label_to_id[getattr(item.label, label_field)] for item in items],
    }
    dataset = Dataset.from_dict(data)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=128),
        batched=True,
    )

## 5. Fine-tune one classifier per task

Two independent `AutoModelForSequenceClassification` fine-tunes (sentiment, event_type) rather than one encoder with two custom heads — more GPU memory for a corpus this tiny doesn't matter, and the standard `Trainer` path has far less custom code to get wrong in a notebook nobody can step through interactively.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments


def fine_tune(label_field: str, label_values: list[str], output_subdir: str):
    train_ds = _to_dataset(train_items, label_field, label_values)
    dev_ds = _to_dataset(dev_items, label_field, label_values)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(label_values)
    )
    args = TrainingArguments(
        output_dir=f"/content/{output_subdir}",
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=5,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        tokenizer=tokenizer,
    )
    trainer.train()
    return trainer


sentiment_trainer = fine_tune("sentiment", SENTIMENTS, "herbert-sentiment")
event_type_trainer = fine_tune("event_type", EVENT_TYPES, "herbert-event-type")

## 6. Evaluate on the held-out test split

REQ-020's metric (macro-F1) computed here for this model only — the full comparison against majority/TF-IDF/few-shot baselines is P4-S7's job (`qa_target.py`, not built yet), not repeated in this notebook.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, f1_score


def evaluate(trainer, label_field: str, label_values: list[str]) -> None:
    test_ds = _to_dataset(test_items, label_field, label_values)
    predictions = trainer.predict(test_ds)
    predicted_ids = np.argmax(predictions.predictions, axis=1)
    true_ids = predictions.label_ids

    macro_f1 = f1_score(true_ids, predicted_ids, average="macro", zero_division=0)
    print(f"{label_field} macro-F1: {macro_f1:.3f}")
    print(
        classification_report(
            true_ids, predicted_ids, target_names=label_values, zero_division=0
        )
    )


evaluate(sentiment_trainer, "sentiment", SENTIMENTS)
evaluate(event_type_trainer, "event_type", EVENT_TYPES)

## 7. End-to-end sample predictions (sentiment + event_type + deterministic tickers)

Ticker matching is the exact same deterministic company-name substring match as `baselines/tfidf_logreg.py::_match_tickers` (not imported directly — that's a private helper of a different module; kept as a small inline copy here rather than reaching into another module's underscore-prefixed internals).

In [ ]:
import torch

from fin_ai_lab.news_classifier.ticker_catalog import TICKER_CATALOG


def match_tickers(text: str) -> list[str]:
    lowered = text.lower()
    return [ticker for ticker, name in TICKER_CATALOG.items() if name.lower() in lowered]


def predict_one(headline_text: str) -> dict:
    inputs = tokenizer(headline_text, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        sentiment_id = sentiment_trainer.model(**inputs).logits.argmax(dim=1).item()
        event_type_id = event_type_trainer.model(**inputs).logits.argmax(dim=1).item()
    return {
        "sentiment": SENTIMENTS[sentiment_id],
        "event_type": EVENT_TYPES[event_type_id],
        "tickers": match_tickers(headline_text),
    }


for item in test_items[:5]:
    print(item.headline.headline)
    print("  predicted:", predict_one(_text(item)))
    print("  teacher  :", item.label)

## 8. Save checkpoints to Drive

Checkpoints never go into the git repo (gitignored, `models/` and `checkpoints/`) — they stay on Drive.

In [ ]:
sentiment_trainer.save_model(f"{CHECKPOINT_DIR}/sentiment")
event_type_trainer.save_model(f"{CHECKPOINT_DIR}/event_type")
print(f"Saved to {CHECKPOINT_DIR}")

## Next steps

- P4-S5: LoRA/QLoRA on Bielik (generative decoder), same corpus, separate notebook.
- P4-S6: load these checkpoints back on the local CPU (no AVX2, no GPU — `docs/ENVIRONMENT.md`), quantize, measure real latency p50/p95 — never assume Colab's numbers transfer.
- P4-S7: `qa_target.py` compares this model against majority/TF-IDF/few-shot on the same test split.